# 01 · Data understanding

**Goal of the project:** predict who wins a UFC fight using only what is known *before* the fight.

Before cleaning anything, we answer three questions:
1. What files do we have and how do they relate?
2. What does every column mean, and **when is it known** (before or after the fight)?
3. What is broken (missing rows, encoding issues, duplicate names)?

In [1]:
import os, sys
from pathlib import Path
# Run from the project root no matter where Jupyter was started.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)

In [2]:
raw = Path("data/raw")
large   = pd.read_csv(raw / "large_dataset.csv")
medium  = pd.read_csv(raw / "medium_dataset.csv")
fstats  = pd.read_csv(raw / "fighter_stats.csv")
events  = pd.read_csv(raw / "completed_events_small.csv")
for name, df in [("large", large), ("medium", medium), ("fighter_stats", fstats), ("events", events)]:
    print(f"{name:14s} {df.shape}")

large          (7439, 95)
medium         (7582, 19)
fighter_stats  (2479, 16)
events         (683, 3)


## The large dataset: one row per fight, 95 columns
Red and blue are the two corners. The red corner is usually the favorite or the champion.

In [3]:
large.head(3).T.head(40)

,0,1,2
event_name,UFC Fight Night: Ribas vs. Namajunas,UFC Fight Night: Ribas vs. Namajunas,UFC Fight Night: Ribas vs. Namajunas
r_fighter,Amanda Ribas,Karl Williams,Edmen Shahbazyan
b_fighter,Rose Namajunas,Justin Tafa,AJ Dobson
winner,Blue,Red,Red
weight_class,Women's Flyweight,Heavyweight,Middleweight
is_title_bout,0,0,0
gender,Women,Men,Men
method,Decision - Unanimous,Decision - Unanimous,KO/TKO
finish_round,5,3,1
total_rounds,5.0,3.0,3.0


## A data dictionary, sorted by *when the value is known*

This is the most important table in the project. Each column falls into one of these groups:

| Group | Known before the fight? | Use for prediction? |
|---|---|---|
| `identity` | yes | no (names aren't features) |
| `context` | yes | yes |
| `physical snapshot` | ~yes (height/reach rarely change) | yes, with care |
| `career snapshot` | **no**: values as of the 2024 scrape | ❌ leakage |
| `in-fight stats` | **no**: produced by the fight itself | ❌ leakage (but great for building *history*) |
| `outcome` / `target` | no | target only |

In [4]:
IN_FIGHT = ["kd","sig_str","sig_str_att","sig_str_acc","str","str_att","str_acc","td","td_att",
            "td_acc","sub_att","rev","ctrl_sec"]
CAREER = ["wins_total","losses_total","age","weight","SLpM_total","SApM_total","sig_str_acc_total",
          "td_acc_total","str_def_total","td_def_total","sub_avg","td_avg"]
PHYSICAL = ["height","reach","stance"]

def group_of(col):
    base = col[2:] if col[:2] in ("r_","b_") else col.replace("_diff","")
    if col in ("event_name","r_fighter","b_fighter","referee"): return "identity"
    if col == "winner": return "target"
    if col in ("method","finish_round","time_sec"): return "outcome"
    if col in ("weight_class","is_title_bout","gender","total_rounds"): return "context"
    if base in IN_FIGHT: return "in-fight stats"
    if base in CAREER: return "career snapshot"
    if base in PHYSICAL: return "physical snapshot"
    return "?"

dictionary = pd.DataFrame({"column": large.columns,
                           "group": [group_of(c) for c in large.columns],
                           "dtype": large.dtypes.astype(str).values,
                           "pct_missing": (large.isna().mean()*100).round(1).values,
                           "example": large.iloc[0].values})
print(dictionary.group.value_counts())
dictionary

group
in-fight stats       39
career snapshot      36
physical snapshot     8
identity              4
context               4
outcome               3
target                1
Name: count, dtype: int64


,column,group,dtype,pct_missing,example
0,event_name,identity,object,0.0,UFC Fight Night: Ribas vs. Namajunas
1,r_fighter,identity,object,0.0,Amanda Ribas
2,b_fighter,identity,object,0.0,Rose Namajunas
3,winner,target,object,0.0,Blue
4,weight_class,context,object,0.0,Women's Flyweight
...,...,...,...,...,...
90,td_acc_total_diff,career snapshot,float64,0.0,0.04
91,str_def_total_diff,career snapshot,float64,0.0,-0.02
92,td_def_total_diff,career snapshot,float64,0.0,0.26
93,sub_avg_diff,career snapshot,float64,0.0,0.2


### Proof that the "career" columns are a 2024 snapshot
If `r_age` were age *at the fight*, a fighter's age would change across fights, and the average would be about 30.
Instead it's about 38, and Jon Jones's `wins_total` is the same number in his 2009 fight and in his 2023 fight.

In [5]:
print("mean r_age:", large.r_age.mean().round(1))
jj = pd.concat([large[large.r_fighter=="Jon Jones"][["event_name","r_wins_total","r_age"]]
                  .rename(columns={"r_wins_total":"wins_total","r_age":"age"}),
                large[large.b_fighter=="Jon Jones"][["event_name","b_wins_total","b_age"]]
                  .rename(columns={"b_wins_total":"wins_total","b_age":"age"})])
jj.tail(4)

mean r_age: 38.3


,event_name,wins_total,age
6488,UFC 87: Seek And Destroy,27,36.0
5874,UFC 128: Shogun vs Jones,27,36.0
6138,UFC Live: Vera vs Jones,27,36.0
6201,The Ultimate Fighter: Heavyweights Finale,27,36.0


**Takeaway:** a model that uses `wins_total_diff` for a 2010 fight already knows how the next 14 years went.
That inflates accuracy and is useless for predicting tomorrow's fight. In notebook 04 we measure *how much* it inflates.

## Missing values

In [6]:
miss = large.isna().sum()
miss[miss > 0].sort_values(ascending=False)

reach_diff      1038
b_reach          888
r_reach          412
age_diff         213
b_age            190
r_age             76
b_stance          68
referee           32
total_rounds      31
r_stance          26
dtype: int64

- `reach` is missing much more often for blue than red (debutants and older-era fighters are usually blue). We'll keep a **missing flag** instead of quietly filling the gap.
- `total_rounds` is missing for 31 fights. Those are the earliest UFCs, which had no rounds at all.

## The medium dataset: it has the dates

In [7]:
print(medium.status.value_counts(), "\n")
broken = medium[medium.event.isna()]
print("rows with status 'not properly finished':", len(broken),
      "| non-empty columns in them:", list(broken.columns[broken.notna().any()]))

status
win                                7439
Fight was not properly finished      85
draw                                 58
Name: count, dtype: int64 

rows with status 'not properly finished': 85 | non-empty columns in them: ['status']


The 85 "not properly finished" rows are **completely empty** (failed scrapes), so we drop them.
The 58 draws exist only in the medium file. We keep them for each fighter's *history* (experience), but they can't be prediction targets.

## Joining large ↔ medium to get a date for every fight
A fight is identified by event + the two fighter names (sorted, so corner order doesn't matter).

In [8]:
key = lambda e, a, b: e + "|" + "|".join(sorted([a, b]))
m = medium.dropna(subset=["event"])
L_keys = {key(e, a, b) for e, a, b in zip(large.event_name, large.r_fighter, large.b_fighter)}
M_keys = {key(e, a, b) for e, a, b in zip(m.event, m.r_fighter, m.b_fighter)}
unmatched = [k for k in L_keys if k not in M_keys]
print(f"large fights matched: {1 - len(unmatched)/len(L_keys):.2%}  ({len(unmatched)} unmatched)")
print({k.split('|')[0] for k in unmatched})
print([e for e in m.event.unique() if "Team Joanna" in e])

large fights matched: 99.84%  (12 unmatched)
{'The Ultimate Fighter: Team Joanna vs. Team Cláudia Finale'}
['The Ultimate Fighter: Team Joanna vs. Team ClГЎudia Finale']


All 12 unmatched fights come from one event. The medium file shows **mojibake**: "Cláudia" was saved as UTF‑8 and then read back as Cyrillic (cp1251), so it became "ClГЎudia".
`ufc.cleaning.fix_mojibake` reverses that, and `norm_name` strips accents so the join is robust.

In [9]:
from ufc.cleaning import fix_mojibake, norm_name
bad = [e for e in m.event.unique() if "Team Joanna" in e][0]
print(bad, "->", fix_mojibake(bad), "->", norm_name(bad))

The Ultimate Fighter: Team Joanna vs. Team ClГЎudia Finale -> The Ultimate Fighter: Team Joanna vs. Team Cláudia Finale -> the ultimate fighter: team joanna vs. team claudia finale


## Hidden leakage #3: before 2010, "red corner" means "winner"
Look at the red-corner win rate by year:

In [10]:
yr = large.merge(m.assign(date=pd.to_datetime(m.date))[["event","date"]].drop_duplicates("event"),
                 left_on="event_name", right_on="event", how="left")
yr["red_win"] = (yr.winner == "Red").astype(int)
print(yr.groupby(yr.date.dt.year).red_win.mean().round(2).to_string())
print("first blue win:", yr.loc[yr.red_win == 0, "date"].min().date())

date
1994.0    1.00
1995.0    1.00
1996.0    1.00
1997.0    1.00
1998.0    1.00
1999.0    1.00
2000.0    1.00
2001.0    1.00
2002.0    1.00
2003.0    1.00
2004.0    1.00
2005.0    1.00
2006.0    1.00
2007.0    1.00
2008.0    1.00
2009.0    1.00
2010.0    0.61
2011.0    0.60
2012.0    0.62
2013.0    0.59
2014.0    0.63
2015.0    0.57
2016.0    0.59
2017.0    0.56
2018.0    0.54
2019.0    0.57
2020.0    0.61
2021.0    0.57
2022.0    0.59
2023.0    0.57
2024.0    0.54
first blue win: 2010-03-21


Red wins **100%** of fights through February 2010. That isn't a real corner advantage: for older events, ufcstats records the winner in the first (red) slot.
A model trained on those rows learns "red always wins", which is the answer key hidden in the column order.

Fix (applied in cleaning): flag `corners_reliable = date >= 2010-03-21`. Pre-2010 fights still count toward each fighter's **history** (wins, strikes, experience), but they are **not used as prediction targets**.

**Lesson:** always check the target against time, source, and every other "harmless" grouping. Leakage hides in data-collection quirks, not just in obvious columns.

## Fighter stats: duplicate names
Name is our only fighter ID, so duplicate names matter.

In [11]:
fstats[fstats.name.duplicated(keep=False)].sort_values("name")[["name","wins","losses","height","weight"]]

,name,wins,losses,height,weight
382,Bruno Silva,23.0,10.0,182.88,83.91
638,Bruno Silva,13.0,5.0,162.56,56.70
1879,Edward Faaloloto,2.0,5.0,185.42,70.31
1897,Edward Faaloloto,2.0,5.0,177.80,70.31
2410,Felix Lee Mitchell,2.0,7.0,182.88,86.18
2419,Felix Lee Mitchell,2.0,7.0,190.50,102.06
2462,Felix Lee Mitchell,2.0,7.0,185.42,92.99
2390,Jack Nilson,2.0,6.0,182.88,90.72
2408,Jack Nilson,2.0,6.0,180.34,104.33
2383,Sam Fulton,0.0,2.0,175.26,102.51


Only **Bruno Silva** is a real collision: a middleweight (84 kg) and a flyweight (57 kg). The rest are repeat entries for 1990s fighters.
The cleaning step splits the two Bruno Silvas by weight.

## Summary
- Usable fights: 7,439 decided + 58 draws, dated **1994‑03 → 2024‑03**.
- Most of the 95 columns are **leakage** (in-fight stats or a 2024 career snapshot).
- Corner order before 2010‑03 is leakage too (red = winner).
- Strategy: keep the in-fight stats only to **build each fighter's history from earlier fights**. → `01b_new_data_source.ipynb`